# SCWF first-pass coefficient export

This notebook runs the **first pass** of the SCWF pipeline in coefficient-only mode.

The goal of the first pass is **not** to save spectra or moments.  
It saves only the row-level coefficient store needed for the second-pass conditional analysis.

The row store is written **once on `ell_all`**.  
Directional buckets such as `ell_perp`, `Ell_perp`, and `ell_par` are reconstructed later from the saved local angles `thetas` and `phis`.

By default the magnetic coefficients are saved in **nT-based wavelet units**:
- `nT * s^0.5` for coefficient amplitudes such as `W_B_mag`,
- `nT * s^0.5` for projected amplitudes such as `B_perp`.

If you set `return_B_in_vel_units=True`, those magnetic coefficient columns switch to Alfv\'enic velocity units instead.


In [2]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import importlib.util
import sys
from joblib import Parallel, delayed
import numpy as np
import pandas as pd

def _find_pipeline_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in (current, *current.parents):
        if (candidate / "data_analysis.py").is_file() and (candidate / "three_D_funcs.py").is_file():
            return candidate
    raise RuntimeError("Could not locate the cleaned pipeline root from the notebook location.")

pkg_dir = _find_pipeline_root(Path.cwd())
pipeline_folder = pkg_dir.name
da_file = pkg_dir / "data_analysis.py"

for stale in [
    "data_analysis",
    "three_D_funcs",
    "scwf_anisotropy",
    "scwf_anisotropy.three_D_funcs",
    "scwf_anisotropy.data_analysis",
    pipeline_folder,
    f"{pipeline_folder}.three_D_funcs",
    f"{pipeline_folder}.data_analysis",
    f"{pipeline_folder}_data_analysis",
]:
    sys.modules.pop(stale, None)

da_spec = importlib.util.spec_from_file_location(f"{pipeline_folder}_data_analysis", da_file)
if da_spec is None or da_spec.loader is None:
    raise RuntimeError(f"Could not load pipeline front door from {da_file}")

data_analysis = importlib.util.module_from_spec(da_spec)
da_spec.loader.exec_module(data_analysis)

from functions import general_functions as func

assert hasattr(data_analysis, "run_logscale_filterbank_analysis")

print("pkg_dir =", pkg_dir)
print("da_file =", da_file)


pkg_dir = C:\Users\nokni\work\MHDTurbPy\functions\scwf_cleaned_pkg_rev12
da_file = C:\Users\nokni\work\MHDTurbPy\functions\scwf_cleaned_pkg_rev12\data_analysis.py


In [3]:
# -----------------------------------------------------------------------------
# Interval selection and run options
# -----------------------------------------------------------------------------
sc = "WIND"
lp = rf"C:\Users\nokni\work\WIND_3D\data\3_sec\\"

n_jobs                   = 15
overwrite_existing_files = False

# First pass: save only coefficient rows.
return_flucs             = True

estimate_alignment_angle = False
return_B_in_vel_units    = True  # False -> B columns in nT * s^0.5 (default), True -> km/s * s^0.5
use_local_polarity       = True
consider_Vsc             = False

strict_thresh            = False
extra_conditions         = True
only_general             = 1
thetas_phis_step         = 5

theta_thresh_gen = 0 if only_general == 1 else None
phi_thresh_gen   = 0 if only_general == 1 else None

conditions = {
    "ell_perp":     {"theta": 80, "phi": 80},
    "Ell_perp":     {"theta": 80, "phi": 10},
    "ell_par":      {"theta": 10, "phi": 90},
    "ell_par_rest": {"theta": 10, "phi": 10},
}
if strict_thresh:
    conditions = {
        "ell_perp":     {"theta": 85, "phi": 85},
        "Ell_perp":     {"theta": 85, "phi": 5},
        "ell_par":      {"theta": 5,  "phi": 90},
        "ell_par_rest": {"theta": 5,  "phi": 5},
    }

qorder           = np.arange(1, 8)
wname            = "mw8"
method_token     = "scwf"
output_subdir    = None
file_name_root   = None
max_interval_dur = 240

# -----------------------------------------------------------------------------
# Saved-variable catalog
#
# The pipeline always saves the lean core needed for B-based conditional analysis.
# Every variable listed below is either:
#   - always saved by the lean store, or
#   - optional and added only if you keep save=True.
#
# Set "save" to True only for extras that you really need.
# -----------------------------------------------------------------------------
SAVE_CATALOG = {
    # Required geometric / normalization core
    "l_mag":                    {"save": True,  "required": True,  "note": "Required. Overall local scale in d_i; use for ell_all reductions."},
    "l_ell":                    {"save": True,  "required": True,  "note": "Required. Parallel projected local scale in d_i; use for ell_par reductions."},
    "l_xi":                     {"save": True,  "required": True,  "note": "Required. Displacement-direction projected local scale in d_i; use for Ell_perp reductions."},
    "l_lambda":                 {"save": True,  "required": True,  "note": "Required. Perpendicular-direction projected local scale in d_i; use for ell_perp reductions."},
    "thetas":                   {"save": True,  "required": True,  "note": "Required. Enough to reconstruct directional buckets later."},
    "phis":                     {"save": True,  "required": True,  "note": "Required. Enough to reconstruct directional buckets later."},
    "coi_mask":                 {"save": True,  "required": True,  "note": "Required. Use to exclude samples outside the cone of influence."},
   # "level_index":              {"save": True,  "required": False, "note": "Recommended. Useful for auditing and debugging level-specific behavior."},
    "is_effective_level":       {"save": True,  "required": True,  "note": "Required if you want to drop ineffective levels downstream."},
    "tau_equiv_seconds":        {"save": True,  "required": True,  "note": "Required. Use in |a|^q / tau^(q/2) for scale-normalized higher-order moments."},
    "frequency_hz":             {"save": True,  "required": True,  "note": "Required. Frequency label for conditional spectra."},
    "response_energy_integral": {"save": True,  "required": True,  "note": "Required. Use in |a|^2 / A_j for conditional PSD estimates."},

    # Required B-based trace / component core
    "W_B_mag":                  {"save": True,  "required": True,  "note": "Required. Magnetic trace coefficient magnitude in the chosen B units."},
    "B_perp":                   {"save": True,  "required": True,  "note": "Recommended default. Magnetic perpendicular coefficient magnitude in the chosen B units; nT by default."},

    # Required magnetic compressibility surrogates (time-to-time values)
    "compress_simple":          {"save": True,  "required": True,  "note": "Required. Legacy alias of compress_simple_parallel = |B_parallel|^2 / |B_trace|^2 at each sample."},
    "compress_simple_parallel": {"save": True,  "required": True,  "note": "Required. Time-to-time ratio |B_parallel|^2 / |B_trace|^2."},
    "compress_simple_absB":     {"save": True,  "required": True,  "note": "Required. Time-to-time ratio |delta|B||^2 / |B_trace|^2 from the scalar |B| coefficient."},

    # Optional conditioning variables
    "sig_c_ts":                 {"save": True, "required": False, "note": "Optional. Samplewise conditioning variable for sigma_c."},
    "sig_r_ts":                 {"save": True, "required": False, "note": "Optional. Samplewise conditioning variable for sigma_r."},

    
    "sig_c":                    {"save": False, "required": False, "note": "Optional. Scale-level cross-helicity surrogate."},
    "sig_r":                    {"save": False, "required": False, "note": "Optional. Scale-level residual-energy surrogate."},

   "compress_simple_V":        {"save": False, "required": False, "note": "Optional. Velocity compressibility surrogate."},

    # Optional projected magnetic components
    "B_par":                    {"save": False, "required": False, "note": "Optional. Save only if you want magnetic parallel-component moments/spectra in the chosen B units."},
    "B_xi":                     {"save": False, "required": False, "note": "Optional. Usually not needed for standard B analysis."},
    "B_lambda":                 {"save": False, "required": False, "note": "Optional. Usually not needed for standard B analysis."},

    # Optional velocity / Elsasser traces
    "W_V_mag":                  {"save": True, "required": False, "note": "Optional. Save only if you want trace conditional spectra/moments for V."},
    "W_Zp_mag":                 {"save": True, "required": False, "note": "Optional. Save only if you want trace conditional spectra/moments for Z+."},
    "W_Zm_mag":                 {"save": True, "required": False, "note": "Optional. Save only if you want trace conditional spectra/moments for Z-."},

    # Optional projected velocity / Elsasser components
    "V_par":                    {"save": False, "required": False, "note": "Optional. Needed only for projected velocity-component analysis."},
    "V_perp":                   {"save": False, "required": False, "note": "Optional. Needed only for projected velocity-component analysis."},
    "V_xi":                     {"save": False, "required": False, "note": "Optional. Needed only for projected velocity-component analysis in the local xi direction."},
    "V_lambda":                 {"save": False, "required": False, "note": "Optional. Needed only for projected velocity-component analysis in the local lambda direction."},
    "Zp_par":                   {"save": False, "required": False, "note": "Optional. Needed only for projected Z+ analysis."},
    "Zp_perp":                  {"save": False, "required": False, "note": "Optional. Needed only for projected Z+ analysis."},
    "Zp_xi":                    {"save": False, "required": False, "note": "Optional. Needed only for projected Z+ analysis."},
    "Zp_lambda":                {"save": False, "required": False, "note": "Optional. Needed only for projected Z+ analysis."},
    "Zm_par":                   {"save": False, "required": False, "note": "Optional. Needed only for projected Z- analysis."},
    "Zm_perp":                  {"save": False, "required": False, "note": "Optional. Needed only for projected Z- analysis."},
    "Zm_xi":                    {"save": False, "required": False, "note": "Optional. Needed only for projected Z- analysis."},
    "Zm_lambda":                {"save": False, "required": False, "note": "Optional. Needed only for projected Z- analysis."},

    # Optional metadata / diagnostics
    "polarity":                 {"save": False, "required": False, "note": "Optional. Save only if you want to condition explicitly on global polarity."},
    "local_polarity":           {"save": False, "required": False, "note": "Optional. Save only if you want to condition explicitly on local polarity."},
    "polarity_used":            {"save": False, "required": False, "note": "Optional. Debug / audit field for polarity handling."},
    "tau_equiv_samples":        {"save": False, "required": False, "note": "Optional. Sample-count version of tau_equiv_seconds."},
    "scale_seconds":            {"save": False, "required": False, "note": "Optional. Filter scale in seconds; useful only for diagnostics."},
    "scale_samples":            {"save": False, "required": False, "note": "Optional. Filter scale in samples; useful only for diagnostics."},
    "period_s":                 {"save": False, "required": False, "note": "Optional. Period label. You can derive this from frequency_hz."},
    "bandwidth_hz":             {"save": False, "required": False, "note": "Optional. Useful only for auditing the filter response."},
}

# The lean geometric / normalization / magnetic core is always saved.
# Every other variable is requested here on purpose so you can drop what you do not need.
ts_list = [key for key, meta in SAVE_CATALOG.items() if meta["save"]]

catalog_df = pd.DataFrame(
    [
        {
            "column": key,
            "save": meta["save"],
            "required": meta["required"],
            "note": meta["note"],
        }
        for key, meta in SAVE_CATALOG.items()
    ]
)
catalog_df


,column,save,required,note
0,l_mag,True,True,Required. Overall local scale in d_i; use for ...
1,l_ell,True,True,Required. Parallel projected local scale in d_...
2,l_xi,True,True,Required. Displacement-direction projected loc...
3,l_lambda,True,True,Required. Perpendicular-direction projected lo...
4,thetas,True,True,Required. Enough to reconstruct directional bu...
5,phis,True,True,Required. Enough to reconstruct directional bu...
6,coi_mask,True,True,Required. Use to exclude samples outside the c...
7,is_effective_level,True,True,Required if you want to drop ineffective level...
8,tau_equiv_seconds,True,True,Required. Use in |a|^q / tau^(q/2) for scale-n...
9,frequency_hz,True,True,Required. Frequency label for conditional spec...


The catalog above is the column contract for the first-pass row store.

A few practical points:

- `thetas` and `phis` are enough to reconstruct the directional buckets later.  
  You do **not** need to save boolean bucket flags.
- By default the magnetic coefficients are saved in **nT** units because `return_B_in_vel_units=False`.
- `compress_simple_parallel` and `compress_simple_absB` are **samplewise** ratios.  
  They are the right quantities to save if you want later conditioning on magnetic compressibility.
- `compress_simple` is kept as a backward-compatible alias of `compress_simple_parallel`.
- Save projected V or Elsasser columns only if you really need them. They increase both CPU cost and file size.


In [4]:
fnames = func.load_files(lp, "final.pkl")
len(fnames)


C:\Users\nokni\work\WIND_3D\data\3_sec\*\final.pkl


412

In [5]:
results = Parallel(n_jobs=n_jobs)(
    delayed(data_analysis.run_logscale_filterbank_analysis)(
        i=i,
        fnames=fnames,
        credentials=None,
        conditions=conditions,
        return_flucs=return_flucs,
        consider_Vsc=consider_Vsc,
        Estimate_5point=False,
        keep_wave_coeefs=False,
        strict_thresh=int(strict_thresh),
        max_hours=300.0,
        qorder=qorder,
        estimate_alignment_angle=estimate_alignment_angle,
        return_mag_align_correl=False,
        only_general=only_general,
        phi_thresh_gen=phi_thresh_gen,
        theta_thresh_gen=theta_thresh_gen,
        sc=sc,
        extra_conditions=extra_conditions,
        ts_list=ts_list,
        overwrite_existing_files=overwrite_existing_files,
        thetas_phis_step=thetas_phis_step,
        return_B_in_vel_units=return_B_in_vel_units,
        max_interval_dur=max_interval_dur,
        use_local_polarity=use_local_polarity,
        wname=wname,
        output_subdir=output_subdir,
        file_name_root=file_name_root,
        method_token=method_token,
    )
    for i in range(145,  len(fnames))
)


In [ ]:
# Inspect the saved coefficient schema from the first successful result
first_ok = next((r for r in results if isinstance(r, dict) and r.get("CoefficientStore") is not None), None)
if first_ok is None:
    raise RuntimeError("No coefficient store was returned.")

store = first_ok["CoefficientStore"]
print("store version:", store["version"])
print("b_export_units:", store["b_export_units"])
print("n columns:", len(store["column_order"]))
print(store["column_order"])


The next step is to use `cond_averaging_scwf.ipynb` on the saved `CoefficientStore` pickles.

Second-pass formulas:

- conditional PSD from a coefficient column `a`:
  \[
  \widehat{P} = \left\langle \frac{|a|^2}{A_j} \right\rangle,
  \qquad
  A_j = \texttt{response\_energy\_integral},
  \]
- scale-normalized higher-order moment:
  \[
  \widehat{M}_q = \left\langle \frac{|a|^q}{\tau_{\mathrm{eq}}^{q/2}} \right\rangle,
  \qquad
  \tau_{\mathrm{eq}} = \texttt{tau\_equiv\_seconds}.
  \]

Use the same scalar trace observable across directional buckets for **wave-vector anisotropy**.  
Use projected component columns only if you want **component scaling**.
